<a href="https://colab.research.google.com/github/mariya7040ai/-arabic-english-medqa-uncertainty/blob/dev/Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install and imports
!pip -q install -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 100.1 MB/s eta 0:00:00


In [3]:
# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
#N_FULL = 1000                                 # Day 3 full-run size
SEED = 42

In [15]:
# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
N_FULL = 3000                                 # Day 3 full-run size
SEED = 42

In [11]:
"""# 1b. CONFIG: edit only this cell
LANG = "ar"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
#MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MODEL_NAME = "inceptionai/jais-adapted-7b-chat"
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
N_FULL = 1000                                 # Day 3 full-run size
SEED = 42"""

In [16]:
# 2. Seeds and logging
import os, json, time, random, datetime
import numpy as np, pandas as pd, torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LETTERS = list("ABCDEF")

def log_run(text, path="RUN_LOG.md"):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"- {ts}: {text}\n")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

torch 2.11.0+cu130 | cuda: True


In [17]:
# 3. Functions to load MedAraBench (columns as in your Day 1 output)
def load_medarabench(path):
    df = pd.read_csv(path)
    opt_cols = [f"Option {l}" for l in LETTERS]
    rows, bad = [], {"empty_question": 0, "answer_not_a_letter": 0,
                     "answer_points_to_empty_option": 0, "fewer_than_2_options": 0}
    for i, r in df.iterrows():
        q = r["Question"]
        if not isinstance(q, str) or not q.strip():
            bad["empty_question"] += 1; continue
        opts, letters = [], []
        for l, c in zip(LETTERS, opt_cols):
            v = r[c]
            if isinstance(v, str) and v.strip():
                opts.append(v.strip()); letters.append(l)
        ans = r["Correct Answer"]
        if not isinstance(ans, str) or ans.strip().upper() not in LETTERS:
            bad["answer_not_a_letter"] += 1; continue
        ans = ans.strip().upper()
        if ans not in letters:
            bad["answer_points_to_empty_option"] += 1; continue
        if len(opts) < 2:
            bad["fewer_than_2_options"] += 1; continue
        rows.append(dict(id=int(i), question=q.strip(), options=opts,
                         answer_idx=letters.index(ans),
                         specialty=str(r["umbrella_specialty"]), level=str(r["Level"])))
    return pd.DataFrame(rows), bad

In [20]:
# 4. Helpers: convert a table to the common format, and sample questions
def df_to_schema(df, q_col, opt_cols, ans_col, answer_is_index=False, specialty_col=None, level_col=None):
    rows = []
    for i, r in df.iterrows():
        opts = [str(r[c]).strip() for c in opt_cols if isinstance(r[c], str) and str(r[c]).strip()]
        a = r[ans_col]
        idx = int(a) if answer_is_index else LETTERS.index(str(a).strip().upper())
        if len(opts) < 2 or idx >= len(opts):
            continue
        rows.append(dict(id=int(i), question=str(r[q_col]).strip(), options=opts, answer_idx=idx,
                         specialty=str(r[specialty_col]) if specialty_col else "NA",
                         level=str(r[level_col]) if level_col else "NA"))
    return pd.DataFrame(rows)

def sample_df(df, n, seed=SEED):
    return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

In [19]:
# 4b. Load the data for the language chosen in CONFIG
if LANG == "ar":
    data_df, bad = load_medarabench(MEDARA_CSV)
    print("dropped:", bad)
else:
    from datasets import load_dataset
    en_raw = load_dataset("openlifescienceai/medmcqa", split="validation").to_pandas()
    print(en_raw.shape, en_raw.columns.tolist())
    print("cop values:\n", en_raw["cop"].value_counts())
    if "choice_type" in en_raw.columns:
        print("choice_type values:\n", en_raw["choice_type"].value_counts())
    spec = "subject_name" if "subject_name" in en_raw.columns else None
    data_df = df_to_schema(en_raw[en_raw["cop"] >= 0], "question", ["opa", "opb", "opc", "opd"], "cop",
                           answer_is_index=True, specialty_col=spec)
K = data_df["options"].map(len)
print(LANG, "usable questions:", len(data_df))
print("options per question:\n", K.value_counts().sort_index())
print("chance accuracy (mean of 1/K): %.3f" % (1 / K).mean())
print("answer position distribution:\n", data_df["answer_idx"].value_counts(normalize=True).sort_index().round(3))
print("specialties:\n", data_df["specialty"].value_counts().head(25))
log_run(f"Loaded {LANG} data: {len(data_df)} usable questions, chance acc {(1/K).mean():.3f}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

(4183, 11) ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']
cop values:
 cop
0    1348
1    1085
2     925
3     825
Name: count, dtype: int64
choice_type values:
 choice_type
single    2816
multi     1367
Name: count, dtype: int64
en usable questions: 4183
options per question:
 options
4    4183
Name: count, dtype: int64
chance accuracy (mean of 1/K): 0.250
answer position distribution:
 answer_idx
0    0.322
1    0.259
2    0.221
3    0.197
Name: proportion, dtype: float64
specialties:
 specialty
Dental                          1318
Surgery                          369
Pathology                        337
Medicine                         295
Pharmacology                     243
Pediatrics                       234
Anatomy                          234
Gynaecology & Obstetrics         224
Physiology                       171
Biochemistry                     171
Social & Preventive Medicine     129
Microbiology                   

In [ ]:
# 5. Load ONE model in 4-bit (MODEL_NAME comes from the CONFIG cell). Gated models need a Hugging Face token in Colab Secrets (HF_TOKEN).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# from google.colab import userdata; from huggingface_hub import login; login(userdata.get("HF_TOKEN"))

MODEL_TAG = MODEL_NAME.split("/")[-1]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()
print("loaded", MODEL_NAME, "| GPU memory used (GB): %.1f" % (torch.cuda.memory_allocated() / 1e9))
log_run(f"Loaded {MODEL_NAME} in 4-bit on {torch.cuda.get_device_name(0)}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [1]:
# 6. Prompt and option probabilities (one forward pass per question, no text generation)
INSTRUCTION = "Answer the following medical multiple-choice question with the letter of the correct option only.\n\n"

PROMPT_STYLE = "no_prefill"   # options: "no_prefill", "answer_colon", "answer_is"
SUFFIXES = {"no_prefill": "", "answer_colon": "Answer:", "answer_is": "The correct answer is"}

def build_prompt(question, options):
    body = question + "\n" + "\n".join(f"{l}. {o}" for l, o in zip(LETTERS, options))
    user = INSTRUCTION + body
    if tok.chat_template:
        text = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    else:
        text = user + "\n"
    return text + SUFFIXES[PROMPT_STYLE]

def letter_ids(letters):
    out = []
    for l in letters:
        c = set()
        for s in (l, " " + l):
            t = tok.encode(s, add_special_tokens=False)
            if t: c.add(t[-1])
        out.append(sorted(c))
    return out
LETTER_IDS = letter_ids(LETTERS)

@torch.no_grad()
def option_probs(question, options):
    text = build_prompt(question, options)
    enc = tok(text, return_tensors="pt", add_special_tokens=not bool(tok.chat_template)).to(model.device)
    logp = torch.log_softmax(model(**enc).logits[0, -1].float(), dim=-1)
    scores = torch.stack([torch.logsumexp(logp[LETTER_IDS[k]], dim=0) for k in range(len(options))])
    mass = torch.exp(torch.logsumexp(scores, dim=0)).item()      # probability mass the model puts on valid letters
    probs = torch.softmax(scores, dim=0)
    return probs.cpu().numpy(), scores.cpu().numpy(), mass

NameError: name 'LETTERS' is not defined

In [ ]:
PROMPT_STYLE = "no_prefill"   # options: "no_prefill", "answer_colon", "answer_is"
SUFFIXES = {"no_prefill": "", "answer_colon": "Answer:", "answer_is": "The correct answer is"}

def build_prompt(question, options):
    body = question + "\n" + "\n".join(f"{l}. {o}" for l, o in zip(LETTERS, options))
    user = INSTRUCTION + body
    if tok.chat_template:
        text = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    else:
        text = user + "\n"
    return text + SUFFIXES[PROMPT_STYLE]

In [13]:
# 7. Sanity check on 3 questions
test = sample_df(data_df, 3)
for _, r in test.iterrows():
    p, s, m = option_probs(r["question"], r["options"])
    print("K =", len(r["options"]), "| probs:", np.round(p, 3), "| mass on letters: %.2f" % m,
          "| pred:", LETTERS[int(p.argmax())], "| true:", LETTERS[int(r["answer_idx"])])

K = 4 | probs: [1. 0. 0. 0.] | mass on letters: 1.00 | pred: A | true: B
K = 4 | probs: [1. 0. 0. 0.] | mass on letters: 0.99 | pred: A | true: A
K = 4 | probs: [0. 0. 0. 1.] | mass on letters: 1.00 | pred: D | true: A


In [14]:
# 8. Evaluation loop with saving and resume (safe against Colab disconnects)
from tqdm.auto import tqdm

def run_eval(df, tag, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{tag}.jsonl"
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            done = {json.loads(l)["id"] for l in f}
    t0, n_new = time.time(), 0
    with open(path, "a", encoding="utf-8") as f:
        for _, r in tqdm(df.iterrows(), total=len(df)):
            if int(r["id"]) in done:
                continue
            probs, scores, mass = option_probs(r["question"], r["options"])
            rec = dict(id=int(r["id"]), K=len(r["options"]), answer=int(r["answer_idx"]),
                       pred=int(np.argmax(probs)), conf=float(probs.max()), mass=float(mass),
                       probs=[float(x) for x in probs], scores=[float(x) for x in scores],
                       specialty=str(r["specialty"]), level=str(r["level"]))
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); n_new += 1
            if n_new % 25 == 0:
                f.flush()
    secs = time.time() - t0
    print(f"{n_new} new questions in {secs/60:.1f} min ({secs/max(n_new,1):.2f} s per question)")
    return path

# Day 2 test run (N_TEST questions)
sub = sample_df(data_df, N_TEST)
path = run_eval(sub, f"{MODEL_TAG}_{LANG}_seed{SEED}_n{N_TEST}")

  0%|          | 0/50 [00:00<?, ?it/s]

50 new questions in 0.2 min (0.29 s per question)


In [15]:
# 9. Day 2 diagnostics: is the pipeline sane?
def load_results(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])

res = load_results(path)
res["correct"] = res["pred"] == res["answer"]
print("accuracy: %.3f | chance: %.3f" % (res["correct"].mean(), (1 / res["K"]).mean()))
print("mean mass on valid letters: %.3f (should be close to 1; if low, the prompt format needs fixing)" % res["mass"].mean())
print("predicted position distribution:\n", res["pred"].value_counts(normalize=True).sort_index().round(2))
print("true position distribution:\n", res["answer"].value_counts(normalize=True).sort_index().round(2))
log_run(f"Day 2 test {MODEL_TAG} {LANG} n={len(res)}: acc={res['correct'].mean():.3f}, mass={res['mass'].mean():.3f}")

accuracy: 0.560 | chance: 0.250
mean mass on valid letters: 0.888 (should be close to 1; if low, the prompt format needs fixing)
predicted position distribution:
 pred
0    0.18
1    0.36
2    0.22
3    0.24
Name: proportion, dtype: float64
true position distribution:
 answer
0    0.30
1    0.36
2    0.18
3    0.16
Name: proportion, dtype: float64


In [16]:
print(res["mass"].describe().round(3))
print("questions with mass < 0.9:", int((res["mass"] < 0.9).sum()), "of", len(res))
print(res.assign(low=res["mass"] < 0.9).groupby("low")["correct"].agg(["mean", "count"]).round(3))

count    50.000
mean      0.888
std       0.253
min       0.018
25%       0.947
50%       0.989
75%       0.998
max       1.000
Name: mass, dtype: float64
questions with mass < 0.9: 9 of 50
        mean  count
low                
False  0.585     41
True   0.444      9
